# Linear models (OLS, Ridge, Lasso)

Target: `final_grade_pct` (regression). This notebook loads the prepared train/test data saved by `Project.ipynb`, so run that notebook first.

This notebook compares the mean-only baseline, OLS, Ridge and Lasso, following the Week 2 solution lab.

Every model is evaluated on the same 10 outer folds of the training set, using the predictors kept by the Lasso selection step. Each model is fit on the outer training folds, and its validation RMSE is recorded. It is then refitted on all training rows and scored once on the test set. A mean-only baseline is included as a reference.

**What this cell does:** imports the core libraries and loads the prepared train/test data written by `Project.ipynb` (features, target, seed, number of CV folds, and predictor lists from `meta.json`). It prints the train and test shapes as a sanity check.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path('prepared_data')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / 'meta.json') as f:
    meta = json.load(f)

SEED = meta['seed']
N_SPLITS = meta['n_splits']
PREDICTOR_COLS = meta['predictor_cols']
NUMERIC_COLS = meta['numeric_cols']

X_train_final = pd.read_csv(DATA_DIR / 'X_train.csv', index_col=0)
X_test_final = pd.read_csv(DATA_DIR / 'X_test.csv', index_col=0)
Y_train = pd.read_csv(DATA_DIR / 'Y_train.csv', index_col=0)[meta['target']]
Y_test = pd.read_csv(DATA_DIR / 'Y_test.csv', index_col=0)[meta['target']]
print('train:', X_train_final.shape, ' test:', X_test_final.shape)

## Shared setup and mean-only baseline

**What this cell does:** imports the libraries and sets the penalty grids, the inner CV splitter (for tuning) and the 10 outer folds (for comparing models).

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODEL_COLS = list(X_train_final.columns)
ALPHA_GRID = np.logspace(-4, 1, 60)
RIDGE_GRID = np.logspace(-3, 3, 50)
inner_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
outer_folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(X_train_final))

**What this cell does:** `source_variable` maps a one-hot column back to its raw predictor name.

In [ ]:
def source_variable(col):
    # Maps a one-hot column back to its raw predictor name
    for raw in PREDICTOR_COLS:
        if col == raw or col.startswith(raw + '_'):
            return raw
    return col

**What this cell does:** `run_group` computes outer-fold validation RMSE for each model, refits it on all training rows and scores it once on the test set.

In [ ]:
def run_group(factories):
    # factories maps a model name to a function that returns a fresh, unfitted model
    cv_fold_rmse = {name: [] for name in factories}
    for fold, (train_idx, valid_idx) in enumerate(outer_folds, start=1):
        X_tr, X_valid = X_train_final.iloc[train_idx], X_train_final.iloc[valid_idx]
        Y_tr, Y_valid = Y_train.iloc[train_idx], Y_train.iloc[valid_idx]
        for name, make in factories.items():
            model = make().fit(X_tr, Y_tr)
            cv_fold_rmse[name].append(root_mean_squared_error(Y_valid, model.predict(X_valid)))
        print(f'Finished outer fold {fold}/{len(outer_folds)}')

    fitted, rows = {}, []
    for name, make in factories.items():
        model = make().fit(X_train_final, Y_train)
        if isinstance(model, GridSearchCV):
            print(f'{name} selected on all training rows: {model.best_params_}')
            fitted[name] = model.best_estimator_
        else:
            fitted[name] = model
        rows.append({
            'model': name,
            'cv_rmse': np.mean(cv_fold_rmse[name]),
            'cv_rmse_sd': np.std(cv_fold_rmse[name], ddof=1),
            'test_rmse': root_mean_squared_error(Y_test, model.predict(X_test_final)),
        })
    table = pd.DataFrame(rows).set_index('model')
    print(table.round(3).to_string())
    return {'cv_fold_rmse': cv_fold_rmse, 'fitted': fitted, 'table': table}

**What this cell does:** `scaled_model` builds a pipeline that standardizes numeric columns inside each fold, so scaling is fit on training folds only.

In [ ]:
def scaled_model(cols, estimator):
    scale_cols = [c for c in cols if source_variable(c) in NUMERIC_COLS]
    scaling = ColumnTransformer([('scale', StandardScaler(), scale_cols)],
                                remainder='passthrough', verbose_feature_names_out=False)
    return make_pipeline(scaling.set_output(transform='pandas'), estimator)

**What this cell does:** runs the mean-only baseline.

In [ ]:
baseline_results = run_group({'Mean only (baseline)': lambda: DummyRegressor(strategy='mean')})

### Linear models (OLS, Ridge, Lasso)

Ridge and Lasso are standardized inside each fold, and their penalties are chosen by the inner 10-fold CV. OLS uses the same scaling so the three share one pipeline shape.

**What this cell does:** defines the Ridge and Lasso tuners, which grid-search the penalty with the inner 10-fold CV.

In [ ]:
def tune_ridge():
    return GridSearchCV(scaled_model(MODEL_COLS, Ridge()), {'ridge__alpha': RIDGE_GRID},
                        cv=inner_cv, scoring='neg_root_mean_squared_error', n_jobs=-1)


def tune_lasso():
    return GridSearchCV(scaled_model(MODEL_COLS, Lasso(max_iter=20000)), {'lasso__alpha': ALPHA_GRID},
                        cv=inner_cv, scoring='neg_root_mean_squared_error', n_jobs=-1)

**What this cell does:** runs OLS, Ridge and Lasso through `run_group` to get CV RMSE and test RMSE.

In [ ]:
linear_results = run_group({
    'OLS': lambda: scaled_model(MODEL_COLS, LinearRegression()),
    'Ridge': tune_ridge,
    'Lasso': tune_lasso,
})

**What this cell does:** collects each model's coefficients and plots the 20 largest side by side, to show how shrinkage changes them.

In [ ]:
coef_series = {}
for name in ['OLS', 'Ridge', 'Lasso']:
    fitted = linear_results['fitted'][name]
    coef_series[name] = pd.Series(fitted[-1].coef_, index=fitted[:-1].get_feature_names_out())

largest = pd.concat(coef_series, axis=1).abs().max(axis=1)
feature_order = largest.nlargest(min(20, len(largest))).sort_values().index
fig, axes = plt.subplots(1, 3, figsize=(15, 8), sharey=True)
for ax, name in zip(axes, ['OLS', 'Ridge', 'Lasso']):
    ax.barh(feature_order, coef_series[name].loc[feature_order])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{name} coefficients')
    ax.set_xlabel('Grade percentage points')
axes[0].set_ylabel('Predictor')
fig.suptitle('Largest 20 coefficients (numeric predictors standardized; dummy levels vs reference)')
plt.tight_layout()
plt.show()

## Comparison following the Week 2 solution lab

The next cells follow the lab's workflow for comparing the mean-only baseline, OLS, Ridge and Lasso: the OLS summary and correlation check, coefficient uncertainty across resamples, boxplots of the resampled test errors, and paired t-tests. The lab used 20 random 85/15 splits. Here the 10 outer CV folds play that role.

**What this cell does:** standardizes the numeric predictors and fits OLS with `statsmodels`, then prints the full summary (coefficients, standard errors, p-values, 95% CIs) as in the lab.

In [ ]:
import statsmodels.api as sm

Xs_train = (scaled_model(MODEL_COLS, LinearRegression())[:-1].fit(X_train_final)
            .transform(X_train_final).astype(float))
ols_sm = sm.OLS(Y_train, sm.add_constant(Xs_train)).fit()
print(ols_sm.summary())

**What this cell does:** lists the strongest predictor correlations, as the lab did to explain the multicollinearity warning.

In [ ]:
corr = Xs_train.corr()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print('\nStrongest pairwise correlations:')
print(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(8).round(3).to_string())

**What this cell does:** draws the correlation heatmap of the standardized predictors.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=7)
fig.colorbar(im, ax=ax)
ax.set_title('Correlation matrix of predictors')
plt.tight_layout()
plt.show()

**What this cell does:** refits OLS, Ridge and Lasso on each outer training fold and records the coefficients and the penalty each fold selected (the lab did this for Ridge and asked for Lasso as a follow-up). It plots the mean coefficient with error bars of one standard deviation across folds. Wide bars mean the estimate is unstable. For Lasso it also reports how often each coefficient is non-zero.

In [ ]:
fold_coefs = {m: [] for m in ['OLS', 'Ridge', 'Lasso']}
fold_alpha = {'Ridge': [], 'Lasso': []}
makers = {'OLS': lambda: scaled_model(MODEL_COLS, LinearRegression()), 'Ridge': tune_ridge, 'Lasso': tune_lasso}
for train_idx, _ in outer_folds:
    X_tr, Y_tr = X_train_final.iloc[train_idx], Y_train.iloc[train_idx]
    for name, make in makers.items():
        model = make().fit(X_tr, Y_tr)
        if isinstance(model, GridSearchCV):
            fold_alpha[name].append(list(model.best_params_.values())[0])
            model = model.best_estimator_
        fold_coefs[name].append(pd.Series(model[-1].coef_, index=model[:-1].get_feature_names_out()))

**What this cell does:** prints the penalty each outer fold selected for Ridge and Lasso.

In [ ]:
print('Penalty selected in each outer fold:')
print(pd.DataFrame(fold_alpha).round(4).to_string())

**What this cell does:** plots each model's mean coefficient with an error bar of one standard deviation across outer folds. Wide bars mean an unstable estimate.

In [ ]:
coef_frames = {m: pd.DataFrame(v) for m, v in fold_coefs.items()}
order = coef_frames['OLS'].abs().mean().sort_values().index
fig, axes = plt.subplots(1, 3, figsize=(15, 7), sharey=True)
for ax, (name, frame) in zip(axes, coef_frames.items()):
    ax.barh(range(len(order)), frame[order].mean(), xerr=frame[order].std(ddof=1))
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{name}: mean +/- SD over outer folds')
    ax.set_xlabel('Coefficient (standardized numeric predictors)')
axes[0].set_yticks(range(len(order)), order, fontsize=8)
plt.tight_layout()
plt.show()

**What this cell does:** reports how often each Lasso coefficient is non-zero across the outer folds.

In [ ]:
print('Share of outer folds where the Lasso coefficient is non-zero:')
print((coef_frames['Lasso'].abs() > 1e-8).mean().sort_values().round(2).to_string())

**What this cell does:** collects the 10 outer-fold validation RMSEs of the baseline and the three linear models, and draws boxplots to show the spread, not just the mean (the lab's check for noise).

In [ ]:
from itertools import combinations

fold_rmse = {**baseline_results['cv_fold_rmse'], **linear_results['cv_fold_rmse']}
names = list(fold_rmse)
plt.figure(figsize=(7, 4))
plt.boxplot([fold_rmse[n] for n in names])
plt.xticks(range(1, len(names) + 1), names, rotation=15)
plt.ylabel('Validation RMSE (outer folds)')
plt.title('Spread of fold RMSE by model')
plt.tight_layout()
plt.show()

**What this cell does:** runs a paired t-test for every pair of models on the same folds. These tests are exploratory because the folds share training rows.

In [ ]:
pair_rows = [{'pair': f'{a} vs {b}', 'mean_diff': np.mean(fold_rmse[a]) - np.mean(fold_rmse[b]),
              'p_value': ttest_rel(fold_rmse[a], fold_rmse[b]).pvalue} for a, b in combinations(names, 2)]
print(pd.DataFrame(pair_rows).set_index('pair').round(4).to_string())

### Save results

The fold-level errors are saved so `Project-Trees.ipynb` can run its paired tests against OLS and the baseline on the same outer folds.

**What this cell does:** saves the fold-level RMSEs and summary table to `results/linear.json` for `Project-Trees.ipynb`, and prints the combined CV/test table for the baseline and linear models.

In [ ]:
saved = {'cv_fold_rmse': {**baseline_results['cv_fold_rmse'], **linear_results['cv_fold_rmse']},
         'table': {**baseline_results['table'].to_dict(orient='index'),
                   **linear_results['table'].to_dict(orient='index')}}
with open(RESULTS_DIR / 'linear.json', 'w') as f:
    json.dump(saved, f, indent=2)

**What this cell does:** prints the combined CV/test table for the baseline and linear models.

In [ ]:
linear_table = pd.concat([baseline_results['table'], linear_results['table']])
print(linear_table.round(3).to_string())

## Conclusion

The linear models clearly beat the mean-only baseline and are statistically tied with each other.

- **Predictors help.** The mean-only baseline has CV RMSE 11.70. OLS, Ridge and Lasso reach 9.63, a reduction of about 18%, with paired p ≈ 6e-10.
- **Linear models are tied.** OLS, Ridge and Lasso all have CV RMSE 9.63 and fold SD about 0.31. OLS vs Ridge has p = 0.76 and OLS vs Lasso p = 0.46. Ridge's selected penalty (alpha about 11) barely changes the coefficients. The CV-selected Lasso penalty sits at the smallest value on the grid (0.0001), so Lasso is effectively OLS here.
- **Stable across folds.** Ridge's penalty stays between about 8 and 19 in every outer fold, and Lasso keeps all 17 predictors in all 10 folds. Ridge vs Lasso has p = 0.77, so no pair of the three linear models differs.
- **Test set agrees.** Refitted on all training rows and scored once on the 1,800-row test set: mean only 11.71, OLS/Ridge/Lasso 9.83.

**Interpretation.** Previous-year grade, log income, and scholarship status have the largest coefficients, and study hours, internet and computer access, and attendance follow. The tuned penalties do almost nothing, so the signal is close to linear and additive, with little to gain from shrinkage at this sample size.

The tree models are compared against these results in `Project-Trees.ipynb`.